# Navital Gym: quickstart

Deterministic, auto-graded environments for AI agents doing tasks set in India (trip planning with rail quotas, UPI, GST, forms, customer support in Hindi and other languages). By [Navital Labs](https://navitallabs.com).

**Read this first.** Every environment uses a *simplified, documented* version of Indian practice and *synthetic* data. These are tests for AI agents, not statements about current law or railway, bank or tax rules. Not affiliated with or endorsed by NPCI, RBI, GSTN / CBIC, IRCTC / Indian Railways, any bank or any government body. Not financial or tax advice.

This notebook runs top to bottom **without an API key**: only the one cell that calls a hosted model needs a key, and it skips itself if none is set. Nothing here needs more than the Python standard library. About 2 minutes.

1. install, 2. list the envs, 3. see what an agent sees, 4. run the reference solver on 3 tasks, 5. plug in a model (optional), 6. read results per split, 7. regrade a saved run.

In [ ]:
# 1. Install. Inside a checkout of the repo this uses the checkout; elsewhere (Colab) it installs from GitHub.
import importlib.util, os, subprocess, sys

if importlib.util.find_spec("navital_gym") is None:
    here = os.getcwd()
    repo_root = os.path.abspath(os.path.join(here, "..")) if os.path.exists(os.path.join(here, "..", "pyproject.toml")) else None
    target = repo_root or "git+https://github.com/Navitallabs/navital-gym"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", target])

import navital_gym
print("navital_gym imported from", os.path.dirname(navital_gym.__file__))

## 2. What is in the suite

In [ ]:
from navital_gym.cli import list_envs
list_envs()          # env name, number of tasks (how many have no valid answer), and the scripted baselines it ships

## 3. What an agent sees

The agent gets a **prompt** and a set of read-only **tools**. Expected answers, feasibility flags and designer notes are never visible to it (a test enforces that for every task in every env). The world is a fixed synthetic snapshot, so the same call always returns the same thing.

In [ ]:
from navital_gym.envs.trip import load_env as load_trip

trip = load_trip()
obs = trip.reset("IN-TRIP-002")
print(obs["prompt"].split("\n\n")[0])
print()
print("tools:", [t["function"]["name"] for t in obs["tools"]])
print()
trains = trip.world.call("search_trains", {"from_city": "Delhi", "to_city": "Agra", "date": "2026-12-05"})
for t in trains["trains"]:
    print(t["train_id"], t["dep"][-5:], "->", t["arr"][-5:], t["classes"], t["base_fare_pp"])

## 4. Run the reference solver ("oracle") on 3 tasks

The oracle sees hidden task data, so it is an **upper bound, not a fair baseline**. Task 042 has no valid answer (its booking date is outside the booking window), so the right answer is to say so and name the rule.

In [ ]:
from navital_gym.core.runner import run_suite
from navital_gym.envs import trip as trip_pkg

oracle = trip_pkg.get_baselines()["oracle"]
ids = ["IN-TRIP-002", "IN-TRIP-041", "IN-TRIP-042"]
report = run_suite(trip, oracle, ids)
for tid, runs in report["results"].items():
    s = runs[0]
    print(f"{tid}: solved={s.solved}  score={s.score:.2f}  verdict={s.verdict}")
print("pass@1 on these 3:", report["summary"]["pass@1"])

## 5. Plug in a model (optional)

Any model that speaks the OpenAI chat API with tool calling works: hosted, or local (Ollama, vLLM, LM Studio). Put the key in an **environment variable**, never in the notebook. If the variable is not set, this cell prints a note and moves on.

Edit the three settings. Examples: `MODEL_SPEC = "openai:gpt-4o-mini"`; a local server `"ollama:qwen2.5:7b"` (no key needed); any OpenAI-compatible host `"openai:<model>@https://your-host/v1"`. You pay your provider directly, so check its price first. This cell runs only 3 short tasks, and the command-line runner can enforce a dollar limit on your own spend with `--spend-cap-usd`.

In [ ]:
import os, tempfile
from navital_gym.core.agents import make_agent
from navital_gym.envs.form import load_env as load_form

MODEL_SPEC = "openai:gpt-4o-mini"      # provider:model[@base_url]
KEY_ENV = "OPENAI_API_KEY"             # NAME of the environment variable that holds your key
TASKS = ["IN-FORM-001", "IN-FORM-016", "IN-FORM-028"]

provider = MODEL_SPEC.split(":")[0]
needs_key = provider != "ollama"
if needs_key and not os.environ.get(KEY_ENV):
    print(f"Skipping the model run: {KEY_ENV} is not set. Everything else in this notebook already ran without it.")
else:
    if needs_key and provider == "openai" and KEY_ENV != "OPENAI_API_KEY":
        os.environ["OPENAI_API_KEY"] = os.environ[KEY_ENV]          # make_agent reads the provider's standard variable
    form = load_form()
    agent = make_agent(MODEL_SPEC, ledger_path=os.path.join(tempfile.gettempdir(), "navital-ledger.json"))
    rep = run_suite(form, agent, TASKS)
    for tid, runs in rep["results"].items():
        s = runs[0]
        print(f"{tid}: solved={s.solved}  rules failed={[r for r, ok in s.rules.items() if not ok]}")
    print("pass@1:", rep["summary"]["pass@1"])

## 6. Read the results per split

One mixed number is never the headline. Bhasha Support Desk has a **core** split (Hindi and Hinglish) and an **experimental** split (Tamil, Bengali, Marathi; machine-assisted, not native-reviewed). The report prints a table per split and per kind. Below, the reference solver and `noop_lang` (a floor: it never acts, but always replies in the right language) are compared, which shows how much of the score is *not* doing the task.

Read **pass@1** (the share of tasks fully solved) as the headline. The *partial credit* column is a mean of rules passed and still gives a never-acting agent a middling number, so do not compare it across agents as if it were skill.


In [ ]:
from navital_gym.core.runner import format_report
from navital_gym.envs import support as support_pkg

support = support_pkg.load_env(include_experimental=True)
baselines = support_pkg.get_baselines()
reports = [run_suite(support, baselines[name]) for name in ("noop_lang", "oracle")]
print(format_report(reports))
s = reports[1]["summary"]
print()
print("splits in the summary:", sorted(s["by_split"]))
print("oracle pass@1 per split:", {k: round(v["pass@1"], 2) for k, v in s["by_split"].items()})

## 7. Regrade a saved run

Runs are saved with the model's full transcript. After a grader fix you can **regrade** them with the current grader without calling the model again, so every number comes from one commit. The cell below saves a small run in the standard file format, corrupts one stored verdict, and shows that regrading restores it. (For real runs the repo ships `scripts/regrade.py`; this cell does the same thing in a few lines.)

In [ ]:
import json, os, tempfile
from navital_gym.core.agents import extract_json

# a tiny run file in the same layout the CLI writes: scores per task, and transcripts whose last assistant message holds the final JSON answer
form = load_form()
ids = ["IN-FORM-001", "IN-FORM-016"]
solver = __import__("navital_gym.envs.form", fromlist=["x"]).get_baselines()["oracle"]
rep = run_suite(form, solver, ids)
transcripts = {}
for tid in ids:
    obs = form.reset(tid)
    submission = solver.solve(obs, lambda name, args=None: form.world.call(name, args), form.task)
    transcripts[tid] = [[{"role": "user", "content": obs["prompt"]}, {"role": "assistant", "content": json.dumps(submission)}]]
run = {"oracle": {"k": 1, "summary": rep["summary"], "tasks": {t: [s.to_dict() for s in runs] for t, runs in rep["results"].items()}}}

# corrupt one stored verdict, as if an old grader had been wrong
run["oracle"]["tasks"]["IN-FORM-001"][0]["solved"] = False
run["oracle"]["tasks"]["IN-FORM-001"][0]["score"] = 0.0
path = os.path.join(tempfile.mkdtemp(), "form-oracle-demo.json")
json.dump(run, open(path, "w")); json.dump({"oracle": transcripts}, open(path.replace(".json", ".transcripts.json"), "w"))

def regrade(path, env):
    data = json.load(open(path)); tr = json.load(open(path.replace(".json", ".transcripts.json")))
    for agent, d in data.items():
        for tid, runs in d["tasks"].items():
            for i, old in enumerate(runs):
                msgs = tr[agent][tid][i]
                last = [m for m in msgs if m.get("role") == "assistant" and not m.get("tool_calls")][-1]
                env.reset(tid)
                runs[i] = env.grade(env.task, extract_json(last["content"])).to_dict()
    return data

before = json.load(open(path))["oracle"]["tasks"]["IN-FORM-001"][0]["solved"]
after = regrade(path, load_form())["oracle"]["tasks"]["IN-FORM-001"][0]["solved"]
print("IN-FORM-001 stored verdict:", before, "-> after regrading with the current grader:", after)

## Where next

- Per-env READMEs (simplified rules, "rules last checked", sources) live in `navital_gym/envs/<env>/README.md`.
- Run the whole suite from the command line: `navital-gym run --env trip --agent "openai:gpt-4o-mini" --k 3 --out runs/trip.json` (keys from environment variables).
- Adapters for Inspect AI, verifiers and OpenEnv: `pip install "navital-gym[inspect]"` and the others (see the README).
- Found a wrong rule or an unfair task? Please open an issue. Corrections are welcome.

Licences: code Apache-2.0, task data CC-BY-4.0. Navital Labs, [navitallabs.com](https://navitallabs.com).